In [1]:
import numpy as np
from dmipy.core.modeling_framework import MultiCompartmentModel
from dmipy.signal_models import sphere_models, cylinder_models, gaussian_models

from microtool.dmipy import DmipyMultiTissueModel
from microtool.optimize import optimize_scheme
from microtool.utils import IO

print("test change")

/nfs/rtfs03/storage/home/jgroot34/miniforge3/envs/MICROtool_2/lib/python3.8/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


test change


## Setting up the VERDICT model

In [2]:
sphere = sphere_models.S4SphereGaussianPhaseApproximation(diffusion_constant=2.0e-9, diameter=10e-6)
ball = gaussian_models.G1Ball(lambda_iso=2.0e-9)
stick = cylinder_models.C1Stick(mu=[np.pi / 2, np.pi / 2], lambda_par=8.0e-9)

verdict_model = MultiCompartmentModel(models=[sphere, ball, stick])
# In Bonet-Carne et. al. the diameter of the intracellular sphere (d_IC) and the vascular diffusivity (P)
# can be fixed such that we can estimate d_{EES}
verdict_model.set_fixed_parameter('S4SphereGaussianPhaseApproximation_1_diameter', 2.0e-9)
verdict_model.set_fixed_parameter('C1Stick_1_lambda_par', 8.0e-9)

verdict_model.set_fixed_parameter('C1Stick_1_mu', [np.pi / 2, np.pi / 2])

verdict_model = DmipyMultiTissueModel(verdict_model, [.3, .3, .4])
print(verdict_model)

Tissue model with 9 scalar parameters:
Tissue-parameter                                 Value    Scale  Optimize    Fit    Fit Bounds
---------------------------------------------  -------  -------  ----------  -----  ------------------------------------------------
S4SphereGaussianPhaseApproximation_1_diameter   1e-05     1e-06  False       True   (1e-08, 1.9999999999999998e-05)
G1Ball_1_lambda_iso                             2e-09     1e-09  True        True   (1.0000000000000002e-10, 3.0000000000000004e-09)
C1Stick_1_mu_0                                  1.5708    1      False       True   (0.0, 3.141592653589793)
C1Stick_1_mu_1                                  1.5708    1      False       True   (-3.141592653589793, 3.141592653589793)
C1Stick_1_lambda_par                            8e-09     1e-09  False       True   (1.0000000000000002e-10, 3.0000000000000004e-09)
partial_volume_0                                0.3       1      True        True   (0.0, 1.0)
partial_volume_1       

## Setting up the acquisition scheme

In [6]:
from microtool.acquisition_scheme import DiffusionAcquisitionScheme
from microtool.gradient_sampling import sample_uniform_half_sphere

def handle_repeated_parameters(N, scheme):
    repeated_parameters = [
        'DiffusionPulseMagnitude',
        'DiffusionPulseDuration',
        'DiffusionPulseInterval'
    ]

    for name in repeated_parameters:
        param = scheme[name]

        # Check that the b0 measurement is fixed
        if param.optimize_mask[0]:
            raise ValueError(
                f"{name}: b0 measurement at index 0 is still free. "
                "Fix the b0 measurement before setting repetition."
            )

        # Apply repetition period to the remaining free parameters
        param.set_repetition_period(N)


n_directions = 6  # Number of b-vectors per shell
n_shells = 3  # Number of shells

b_vals = np.concatenate([np.repeat(0, n_directions), np.repeat(1000, n_directions), np.repeat(2000, n_directions)])
pulse_durations = np.concatenate([np.repeat(0.019, n_directions), np.repeat(0.016, n_directions), np.repeat(0.007, n_directions)])
pulse_intervals = np.concatenate([np.repeat(0.024, n_directions), np.repeat(0.027, n_directions), np.repeat(0.020, n_directions)])


echo_times = np.repeat(0.050, n_directions * n_shells)  # 50 ms for all measurements

directions = sample_uniform_half_sphere(n_directions * n_shells)
initial_scheme = DiffusionAcquisitionScheme.from_bvals(b_values=b_vals,
                                                       b_vectors=directions,
                                                       pulse_durations=pulse_durations,
                                                       pulse_intervals=pulse_intervals,
                                                       echo_times=echo_times)

initial_scheme.fix_b0_measurements()
handle_repeated_parameters(n_directions, initial_scheme)
initial_scheme["EchoTime"].fixed = True  # Don't optimize the echo time, but keep it fixed to the given value.
#initial_scheme["DiffusionPulseDuration"].fixed = True
#initial_scheme["DiffusionPulseInterval"].fixed = True
print(initial_scheme)

print(initial_scheme.b_values)

Acquisition scheme with 18 measurements and 6 scalar parameters. 
total number of optimized parameters is 6:
  DiffusionPulseMagnitude [mT/mm] in (0.0, 0.528)    DiffusionGradientAnglePhi [rad] (fixed)    DiffusionGradientAngleTheta [rad] (fixed)    DiffusionPulseDuration [s] in (0.001, 0.1)    DiffusionPulseInterval [s] in (0.001, 0.1)    EchoTime [s] (fixed)
-------------------------------------------------  -----------------------------------------  -------------------------------------------  --------------------------------------------  --------------------------------------------  ----------------------
                                       -0                                          0.911669                                      1.5708                                           0.019                                         0.024                    0.05
                                       -0                                         -1.57186                                       

## Optimization

In [7]:
scheme, _ = optimize_scheme(initial_scheme, verdict_model, .02, method="trust-constr", solver_options={"verbose":2,"maxiter":100})
print(scheme)
IO.save_pickle(scheme, 'schemes/optimized_scheme.pkl')

| niter |f evals|CG iter|  obj func   |tr radius |   opt    |  c viol  |
|-------|-------|-------|-------------|----------|----------|----------|
|   1   |   7   |   0   | +1.3783e+01 | 1.00e+00 | 1.16e+01 | 0.00e+00 |


/nfs/rtfs03/storage/home/jgroot34/miniforge3/envs/MICROtool_2/lib/python3.8/site-packages/scipy/optimize/_hessian_update_strategy.py:182: UserWarning: delta_grad == 0.0. Check if the approximated function is linear. If the function is linear better results can be obtained by defining the Hessian as zero instead of using quasi-Newton approximations.
  warn('delta_grad == 0.0. Check if the approximated '


|   2   |  14   |   1   | +1.3783e+01 | 1.00e-01 | 1.16e+01 | 0.00e+00 |
|   3   |  21   |   2   | +7.4399e+00 | 2.00e-01 | 3.14e+00 | 0.00e+00 |
|   4   |  28   |   3   | +4.0040e+00 | 4.00e-01 | 3.53e-01 | 0.00e+00 |
|   5   |  35   |   8   | +3.7237e+00 | 4.11e-01 | 2.47e-01 | 0.00e+00 |
|   6   |  42   |  13   | +3.5486e+00 | 5.96e-01 | 1.98e-01 | 0.00e+00 |
|   7   |  49   |  19   | +3.5404e+00 | 5.96e-01 | 1.90e-01 | 0.00e+00 |
|   8   |  56   |  23   | +3.5402e+00 | 5.96e-01 | 1.89e-01 | 0.00e+00 |
|   9   |  63   |  28   | +3.5398e+00 | 5.96e-01 | 1.71e-01 | 0.00e+00 |
|  10   |  70   |  32   | +3.5382e+00 | 5.96e-01 | 1.71e-01 | 0.00e+00 |
|  11   |  77   |  38   | +3.5169e+00 | 9.76e-01 | 1.82e-01 | 0.00e+00 |
|  12   |  84   |  44   | +3.5055e+00 | 3.08e+00 | 3.25e-01 | 0.00e+00 |
|  13   |  91   |  50   | +3.5062e+00 | 5.49e+00 | 5.19e-01 | 0.00e+00 |
|  14   |  98   |  56   | +3.4391e+00 | 5.49e+00 | 4.87e-01 | 0.00e+00 |
|  15   |  105  |  62   | +3.2003e+00 | 7.04e+00 | 

KeyboardInterrupt: 